# 회의록 만들기

녹음 → 녹취록 → 회의록 → 구글 드라이브. **위에서 아래로 순서대로** 실행합니다.

```
1 설정  →  2 STT  →  3 추출  →  ■ 4 검토(멈춤)  →  5 확정  →  6 업로드
```

4번에서 멈춥니다. 회의에서 나온 말이 전부 결정은 아니므로, **번호로 고른 것만** 문서에 들어갑니다.

> ⚠ 이 노트북의 출력에는 실제 회의 내용이 남습니다. 커밋 전에 `Kernel → Restart & Clear All Outputs` 를 실행하세요.

## 1. 설정

여기만 고치면 됩니다.

In [1]:
# 둘 중 하나만 채웁니다. 둘 다 있으면 AUDIO 가 우선입니다.
AUDIO      = r"data/audio/mom_test1.m4a"   # 녹음 파일. 없으면 "" 로 두세요
TRANSCRIPT = r""                          # 이미 녹취록이 있으면 여기에

TITLE  = "킥오프"          # 회의 제목 (빈 문자열이면 내용에서 자동 생성)
DATE   = "2026-08-25"      # YYYY-MM-DD

UPLOAD = False             # 6번 셀에서 드라이브 업로드 여부

# --- 환경 준비 ---
import os, sys
from pathlib import Path

os.environ.setdefault("PYTHONIOENCODING", "utf-8")
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    raise SystemExit(f"meeting_minutes 폴더에서 열어야 합니다. 현재: {ROOT}")
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from src.config import CFG
CFG.ensure_dirs()

has_key = "있음" if os.getenv("ANTHROPIC_API_KEY") else "없음 — .env 확인"
print(f"모델    {CFG.model}")
print(f"STT     {CFG.whisper_model} ({CFG.language})")
print(f"출력    {CFG.output_dir}")
print(f"API 키  {has_key}")

모델    claude-opus-5
STT     large-v3 (ko)
출력    C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes
API 키  없음 — .env 확인


## 2. STT — 녹음 → 녹취록

GPU 가 없으면 **오래 걸립니다.** 
처음이면 `.env` 의 `WHISPER_MODEL=medium` 으로 5분 클립을 먼저 재보는 걸 권합니다.

녹취록이 이미 있으면 이 셀은 자동으로 건너뜁니다.

In [2]:
if AUDIO:
    from src.transcribe import transcribe
    audio_path = Path(AUDIO)
    assert audio_path.exists(), f"오디오 파일이 없습니다: {audio_path}"
    transcript_path = transcribe(audio_path)
    source_name = audio_path.name
else:
    transcript_path = Path(TRANSCRIPT)
    assert transcript_path.exists(), f"녹취록이 없습니다: {transcript_path}"
    source_name = transcript_path.name
    print(f"STT 건너뜀 — 기존 녹취록 사용: {transcript_path}")

c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[stt] model=large-v3 device=cpu compute=int8


c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\skswl\.cache\huggingface\hub\models--Systran--faster-whisper-large-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[stt] 00:03:13 ...
[stt] 완료: 57 세그먼트, 3.8분
[stt] 저장: C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\transcripts\mom_test1.txt


### 녹취록 확인

추출 전에 눈으로 한 번 봅니다. 여기가 깨져 있으면 뒤가 다 깨집니다.

In [ ]:
transcript = transcript_path.read_text(encoding="utf-8")
lines = transcript.splitlines()
print(f"{len(transcript):,}자 / {len(lines):,}줄")
print("-" * 60)
for l in lines[:15]:
    print(l)
print()
print("... (중략) ...")
print()
for l in lines[-5:]:
    print(l)

## 3. 추출 — Claude Code CLI

API 키를 쓰지 않습니다. `claude -p` 로 Claude Code 를 불러 녹취록을 구조화합니다.

```
셀 ──▶ claude -p ──▶ (녹취록·규칙·스키마 읽기) ──▶ stdout JSON ──▶ 셀이 검증·보정·저장
```

### 실패에서 나온 설계 (실제로 두 번 깨진 뒤 고친 것)

| 시도 | 결과 |
|---|---|
| Claude 가 JSON 파일을 저장 | **실패** — 헤드리스에서 쓰기 권한이 막힘 |
| 사용자 프롬프트로 “JSON만 출력” 지시 | **실패** — 사람용 리포트를 출력 (`-p` 는 포매터가 아니라 에이전트) |
| `--append-system-prompt` 로 형식 강제 | **성공** — 형식 강제는 시스템 프롬프트 층에서 해야 한다 |

그래서 지금 구조는 이렇습니다.

- **읽기만 시킨다.** 파일 쓰기는 셀이 한다 → 권한 확인 프롬프트가 없다
- **형식은 `--append-system-prompt` 로 강제한다**
- **우리가 아는 값은 모델에게 맡기지 않는다.** `TITLE`/`DATE` 는 받은 JSON 위에 셀이 덮어쓴다
  (모델이 제목을 바꾸고 날짜를 `null` 로 두는 것을 실제로 확인했다)
- **규칙·스키마의 정본은 리포에 있다.** 프롬프트에 규칙을 다시 적지 않는다

In [5]:
# ============================================================================
#  [대안] CLI 를 API 처럼 쓰는 방법 — 입력 대체를 «구조적으로» 막는다
# ============================================================================
#  왜 필요한가
#    지금 방식은 녹취록 «경로» 를 넘긴다. CLI 는 디스크 접근권이 있어서
#    읽는 김에 폴더를 훑고, 더 «회의다운» 파일이 있으면 바꿔 읽는다.
#    실제로 mom_test1.txt(가족 통화) 대신 옆에 있던 테스트회의.txt 를 읽었다.
#    악의가 아니라 «선의의 판단» 이라 프롬프트로는 완전히 막기 어렵다.
#
#  API 는 이 사고가 불가능하다
#    extract.py 는 녹취록 «내용» 을 프롬프트에 박아 보낸다.
#    모델은 다른 파일이 있는지조차 모른다 -> 고를 수가 없다.
#
#  CLI 를 그 상태로 만드는 방법: 경로가 아니라 내용을 stdin 으로 넣는다
#
#    r = subprocess.run(
#        [CLAUDE, '-p', prompt_without_path, '--append-system-prompt', SYS],
#        input=transcript,                 # <- 파일이 아니라 텍스트를 직접 준다
#        capture_output=True, text=True, encoding='utf-8', errors='replace',
#        env=CHILD_ENV, cwd=str(ROOT), timeout=3600,
#    )
#
#    프롬프트에서 파일 경로 줄을 빼고 이렇게 바꾼다:
#      '표준입력으로 들어온 텍스트가 녹취록이다. 파일을 읽지 않는다.'
#
#  트레이드오프 (그래서 기본값으로 두지 않았다)
#    - 장점: 폴더를 탐색할 이유가 없어져 입력 대체가 원천 차단된다
#    - 단점: 녹취록 전량이 한 번에 컨텍스트로 들어간다. 파일 경로 방식은
#            필요한 만큼만 읽으며 긴 회의를 나눠 처리할 수 있다.
#    - 단점: 규칙·스키마 파일은 여전히 읽어야 하므로 파일 접근을 아예
#            없앨 수는 없다 (읽기 대상이 줄 뿐이다).
#
#  현재 판단: 경로 방식 + 아래 «인용 검증 가드» 로 잡는다.
#             실패하면 즉시 멈추고 이유를 보여주므로 조용히 틀리지 않는다.
#             긴 회의에서 문제가 생기면 그때 stdin 방식으로 옮긴다.
# ============================================================================

import json, os, re, shutil, subprocess
from datetime import datetime
from src.schema import Minutes, MinutesBundle

CLAUDE = shutil.which('claude')
assert CLAUDE, 'claude CLI 가 없습니다: npm i -g @anthropic-ai/claude-code'
CHILD_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}

# 출력 형식은 시스템 프롬프트 층에서 강제한다. 사용자 프롬프트로는 이기지 못한다.
SYS = (
    '너는 JSON 추출 엔드포인트다. 사람에게 보고하지 않는다.'
    ' 최종 응답은 JSON 객체 하나여야 한다.'
    ' 설명·요약·머리말·꼬리말·코드펜스·마크다운을 절대 쓰지 않는다.'
    ' 응답의 첫 글자는 { 이고 마지막 글자는 } 이다. 파일을 만들지 않는다.'
)

task = [
    '아래 녹취록에서 회의록을 구조화해 Minutes JSON 을 만들어라.',
    '',
    f'녹취록  : {transcript_path}',
    f'회의 제목: {TITLE or "(미지정 - 내용에서 생성)"}',
    f'회의 날짜: {DATE or "(미지정)"}   <- 확정된 사실이다. 상대 날짜(다음 주 월요일 등)는 이 날짜를 기준으로 환산한다.',
    '',
    '읽을 것 (읽기만 한다. 파일을 만들지 않는다):',
    '1. prompts/extract_system.md — 규칙. 그대로 따른다. 새로 만들거나 요약하지 않는다.',
    '2. src/schema.py 의 Minutes — 필드와 허용값. 추측하지 않는다.',
    '3. 위에 지정된 녹취록 «그 파일만».',
    '',
    '입력 고정 (중요):',
    '- 위에 지정된 녹취록 파일만 읽는다. data/transcripts 의 다른 파일을 읽거나 대체하지 않는다.',
    '- 내용이 회의처럼 보이지 않아도(잡담·통화 등) 그 파일을 그대로 처리한다.',
    '  더 «회의다운» 파일을 찾아 바꾸지 않는다. 판단은 사람이 한다.',
    '- 추출할 결정·액션이 없으면 빈 배열로 두고 topics 만 채운다. 다른 파일로 갈아타지 않는다.',
    # 규칙을 여기에 다시 적지 않는다. 두 곳에 있으면 갈라진다 (정본은 prompts/extract_system.md).
]
prompt = chr(10).join(task)

print('추출 중... (녹취록 길이에 따라 몇 분 걸립니다)')
r = subprocess.run(
    [CLAUDE, '-p', prompt, '--append-system-prompt', SYS],
    capture_output=True, text=True, encoding='utf-8', errors='replace',
    env=CHILD_ENV, cwd=str(ROOT), timeout=3600,
)
raw_out = (r.stdout or '').strip()
if r.returncode != 0:
    print(f'exit {r.returncode}')
    print((r.stderr or '')[-800:])


def extract_json(text):
    """응답에서 JSON 본문만 꺼낸다. 코드펜스나 앞뒤 설명이 붙어도 견딘다."""
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.S)
    if m:
        return m.group(1)
    i, j = text.find('{'), text.rfind('}')
    return text[i:j + 1] if (i != -1 and j > i) else None


payload = extract_json(raw_out)
assert payload, ('응답에서 JSON 을 찾지 못했습니다. 받은 내용:' + chr(10) + raw_out[:1200])

data = json.loads(payload)
minutes = Minutes.model_validate(data['minutes'] if 'minutes' in data else data)


def verify_quotes(minutes, transcript):
    """모든 quote 가 «우리가 넘긴» 녹취록에 실제로 있는지 확인한다.

    이 가드는 두 가지를 동시에 잡는다.
      1. 에이전트가 다른 녹취록 파일을 읽어버린 경우 (실제로 발생했다)
      2. 없는 발언을 만들어낸 경우
    공백만 무시하고 부분문자열로 대조한다.
    """
    flat = re.sub(r'\s+', '', transcript)
    bad = []
    for kind, items in (('D', minutes.decisions), ('A', minutes.action_items)):
        for n, it in enumerate(items, 1):
            if re.sub(r'\s+', '', it.quote or '') not in flat:
                bad.append((f'{kind}{n}', it.quote))
    return bad


bad_quotes = verify_quotes(minutes, transcript)
if bad_quotes:
    print('!! 인용 검증 실패 — 녹취록에 없는 발언이 있습니다')
    print(f'   대상 녹취록: {transcript_path}')
    for label, q in bad_quotes:
        print(f'   {label}: {(q or "")[:70]}')
    raise AssertionError(
        '인용이 녹취록과 일치하지 않습니다. 다른 파일을 읽었거나 발언을 만들어낸 것입니다. '
        'data/transcripts 에 다른 파일이 섞여 있는지 확인하고 다시 실행하세요.'
    )
print(f'인용 검증 통과 ({len(minutes.decisions) + len(minutes.action_items)}건 전부 녹취록에 존재)')

# 우리가 아는 값은 모델 결과를 신뢰하지 않는다 (제목 변경·날짜 누락을 실제로 확인).
fix = {}
if TITLE and minutes.title != TITLE:
    print(f'제목 보정: {minutes.title!r} -> {TITLE!r}')
    fix['title'] = TITLE
if DATE and minutes.date != DATE:
    print(f'날짜 보정: {minutes.date!r} -> {DATE!r}')
    fix['date'] = DATE
if fix:
    minutes = minutes.model_copy(update=fix)

bundle = MinutesBundle(
    minutes=minutes,
    source_audio=source_name,
    transcript_chars=len(transcript),
    model='claude-code(cli)',
    generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'),
)

# 셀이 직접 저장한다 (Claude 에게 쓰기 권한을 요구하지 않는다)
draft_dir = CFG.output_dir / 'draft'
draft_dir.mkdir(parents=True, exist_ok=True)
stem = ((DATE + '_') if DATE else '') + (TITLE or transcript_path.stem)
cli_json = draft_dir / (stem + '.cli.json')
cli_json.write_text(json.dumps(bundle.model_dump(), ensure_ascii=False, indent=2), encoding='utf-8')

print()
print(f'검증 통과 · 결정 {len(minutes.decisions)} · 액션 {len(minutes.action_items)} '
      f'· 미결 {len(minutes.open_questions)}')
print(f'저장: {cli_json}')

추출 중... (녹취록 길이에 따라 몇 분 걸립니다)
제목 보정: '병원 챗봇 1차 범위 정의' -> '킥오프'
날짜 보정: None -> '2026-08-25'

검증 통과 · 결정 2 · 액션 3 · 미결 2
저장: C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\draft\2026-08-25_킥오프.cli.json


<details><summary>API 키를 쓰는 경우 (참고)</summary>

`.env` 에 `ANTHROPIC_API_KEY` 가 있으면 위 셀 대신 이걸 쓸 수 있습니다. 분할 추출·병합이 자동입니다.

```python
from datetime import datetime
from src.extract import extract
from src.schema import MinutesBundle

minutes = extract(transcript, title=TITLE or None, date=DATE or None)
bundle = MinutesBundle(minutes=minutes, source_audio=source_name,
                      transcript_chars=len(transcript), model=CFG.model,
                      generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'))
```

</details>

## ■ 4. 검토 — 여기서 멈춥니다

번호를 보고 **무엇을 문서에 반영할지** 고릅니다.

- `<회의에서 안 정해짐>` → 참석자에게 물어야 합니다 (미확인)
- `<녹취 불확실>` → 원본 오디오를 다시 들어야 합니다 (미조사)

In [6]:
from src.review import render_review, blank_report

print(render_review(minutes))

r = blank_report(minutes)
if r["not_stated"]:
    print()
    print(f"! 회의에서 담당/마감을 정하지 않은 액션 {r['not_stated']}건 — 참석자에게 확인")
if r["unclear"]:
    print(f"! 녹취가 불확실해 확인 못 한 액션 {r['unclear']}건 — 원본 오디오 재확인")
if r["notes"]:
    print(f"! 녹취 불확실 구간 {r['notes']}건 — 확정 전 확인 필요")

  킥오프   2026-08-25
  챗봇 1차 오픈 범위를 예약·진료시간·주차 3개로 한정 확정. 외부 LLM API 사용 가능 여부는 정보보안팀 회신 대기로 미해결.

[결정사항]
  D1. 챗봇 1차 범위를 예약·진료시간·주차 3개 도메인으로 한정
      근거: "그럼 일단 그 세 개만 먼저 가는 걸로 하죠." ·00:01:05
  D2. 합성 데이터 문서 40건 제작 담당자 지정은 다음 회의로 보류
      근거: "그건 일단 미정으로 두고 다음 회의에서 정합시다." ·00:01:45

[액션아이템]
  A1. 운영팀에 시나리오 트리 export 를 요청해 수령한다
      담당 <녹취 불확실 · 오디오 재확인> / 마감 <녹취 불확실 · 오디오 재확인> / unknown
  A2. 정보보안팀에 외부 LLM API 사용 가능 여부를 확인한다
      담당 <회의에서 안 정해짐> / 마감 <회의에서 안 정해짐> / high
  A3. 다음 회의에서 합성 데이터 문서 40건 제작 담당자를 확정한다
      담당 <회의에서 안 정해짐> / 마감 <회의에서 안 정해짐> / unknown

[미결 사항]
  Q1. [블로커] 외부 LLM API 를 병원 보안정책상 사용할 수 있는가
  Q2. 합성 데이터 문서 40건은 누가 만드는가

[녹취 불확실 — 확정 전 확인 필요]
  · 회의 날짜가 녹취에 없어 '다음 주 월요일'을 절대 날짜로 환산 불가 — 회의 일자 확인 필요
  · 전사에 발언자 이름이 전혀 없어 '제가 운영팀에 요청'의 담당자를 특정 불가 — 원본 오디오 화자 확인 필요
  · '합성 데이터 문서 40건'의 건수 40 은 STT 오인식 가능 구간 — 사람 확인 필요

------------------------------------------------------------------------
반영할 항목을 고르세요. 회의에서 나온 말이 전부 결정은 아닙니다.
  python -m src.pipeline --accept D1,D2,A

## 5. 확정 — 고른 것만 문서로

위 목록에서 고른 라벨을 적습니다. 전부면 `"all"`.

고르고 나서 마음이 바뀌면 이 셀만 다시 실행하면 됩니다 (파일명이 `_v2` 로 넘어갑니다).

In [ ]:
ACCEPT = "D1,A1,A3"          # 예: "D1,D2,A1,A3"

from src.review import parse_accept, apply_selection
from src.render import render

picked, unknown = parse_accept(ACCEPT, minutes)
assert not unknown, f"알 수 없는 라벨: {unknown}"
assert picked, "선택된 항목이 없습니다"

final = bundle.model_copy(update={"minutes": apply_selection(minutes, picked)})
out = render(final)
print(f"반영 {len(picked)}개: {', '.join(picked)}")
print(f"md   {out.md}")
print(f"html {out.html}")
print(f"json {out.json}")

### 결과 미리보기

In [ ]:
from IPython.display import IFrame, display, Markdown

display(Markdown(out.md.read_text(encoding="utf-8")))

# HTML 뷰어로 보려면 아래 주석 해제 (노트북 폴더 기준 상대경로)
# display(IFrame(src=out.html.relative_to(ROOT).as_posix(), width="100%", height=600))

## 6. 구글 드라이브 업로드

첫 실행 시 브라우저 인증 창이 열립니다 (`credentials.json` 필요). 이후 `token.json` 으로 자동 갱신됩니다.

In [ ]:
if UPLOAD:
    from src.drive import upload_minutes
    for u in upload_minutes(out.md, out.html, out.json, subfolder=out.slug):
        print(f"{u.name}")
        print(f"  {u.link}")
else:
    print("UPLOAD = False — 업로드 건너뜀. 1번 셀에서 True 로 바꾸세요.")

## 7. 노션에 올리기 (Claude Code CLI 경유)

API 토큰을 쓰지 않습니다. `claude -p` 로 Claude Code 를 부르고, **그 Claude 가 노션 커넥터로** 올립니다.

```
셀 ──▶ claude -p ──▶ (노션 커넥터) ──▶ 노션 페이지 생성
```

먼저 헤드리스 실행에서 노션 도구가 붙는지 확인합니다. **대화형 채팅창에서는 붙어도 헤드리스에서는 빠질 수 있습니다.**

In [ ]:
import os, shutil, subprocess

CLAUDE = shutil.which('claude')
CHILD_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
print('claude CLI:', CLAUDE or '없음 - npm i -g @anthropic-ai/claude-code')

NOTION_OK = False
if CLAUDE:
    probe = ('사용 가능한 도구 중 notion 관련 도구의 이름만 쉼표로 나열해. '
             '설명 없이 이름만. 없으면 NONE 만 출력해.')
    try:
        r = subprocess.run([CLAUDE, '-p', probe], capture_output=True, text=True,
                           encoding='utf-8', errors='replace', env=CHILD_ENV, timeout=180)
        out = ((r.stdout or '') + (r.stderr or '')).strip()
        print()
        print(out[:600])
        NOTION_OK = ('notion' in out.lower()) and ('NONE' not in out.upper())
    except subprocess.TimeoutExpired:
        print('시간 초과 - 권한 확인 대기 중일 수 있습니다')

print()
print('헤드리스에서 노션 도구:', '사용 가능' if NOTION_OK else '없음')
if not NOTION_OK:
    print('  -> claude.ai 설정 > 커넥터에서 Notion 을 인증하세요.')
    print('  -> 인증해도 헤드리스에서 안 붙으면 아래 셀이 대체 방법을 안내합니다.')

### 올리기

`NOTION_TARGET` 에 노션의 상위 페이지나 데이터베이스 **이름**을 적습니다 (URL 도 됩니다).

프롬프트가 파일 경로만 넘기므로, 회의 내용이 명령줄에 노출되지 않습니다. Claude 가 파일을 직접 읽습니다.

In [ ]:
NOTION_TARGET = '회의록'        # 노션의 상위 페이지/DB 이름 또는 URL

task = [
    f'다음 회의록 파일을 읽고 노션에 페이지로 올려줘.',
    f'파일: {out.md}',
    f'구조 데이터(참고용): {out.json}',
    f'올릴 위치: 노션의 {NOTION_TARGET}',
    '',
    '규칙:',
    '- 파일에 있는 내용만 쓴다. 요약하거나 새로 만들지 않는다.',
    '- 담당/마감이 <회의에서 안 정해짐> 또는 <녹취 불확실>이면 그 표기를 그대로 유지한다.',
    '- 액션아이템은 체크박스(to-do) 블록으로 만든다.',
    '- 결정사항은 근거 인용을 함께 남긴다.',
    '- 다 끝나면 만든 페이지 URL 한 줄만 마지막에 출력한다.',
]
prompt = chr(10).join(task)

if not NOTION_OK:
    print('헤드리스에서 노션 도구가 없습니다. 채팅창에 아래를 붙여넣으세요:')
    print()
    print(prompt)
else:
    print('노션 업로드 중... (몇 분 걸릴 수 있습니다)')
    r = subprocess.run([CLAUDE, '-p', prompt], capture_output=True, text=True,
                       encoding='utf-8', errors='replace', env=CHILD_ENV, timeout=900)
    res = ((r.stdout or '') + (r.stderr or '')).strip()
    print(res[-2000:])
    if r.returncode != 0:
        print()
        print(f'실패 (exit {r.returncode})')
        print('권한 확인에서 멈춘 경우: 아래처럼 도구를 미리 허용하고 다시 시도하세요.')
        print("  subprocess.run([CLAUDE, '-p', prompt, '--allowedTools', 'mcp__notion'])")

---

## 다시 돌릴 때

| 하고 싶은 것 | 실행할 셀 |
|---|---|
| 고르는 항목만 바꾸기 | 5번만 (`ACCEPT` 수정) |
| 추출 품질이 아쉬움 | `prompts/extract_system.md` 고치고 3번부터 |
| 다른 회의 | 1번부터 (`AUDIO`/`TITLE`/`DATE` 수정) |
| STT 다시 | 2번부터 (`.env` 의 `WHISPER_MODEL` 조정) |

**커밋 전에 `Kernel → Restart & Clear All Outputs`** — 출력에 회의 내용이 남아 있습니다.